# AI Explainer - Colab Workflow (Fish Speech + Whisper + SDXL)

This notebook runs the entire middle pipeline on Google Colab:
1. **Voice Generation:** Uses Fish Speech (Zero-shot Voice Cloning)
2. **Captioning:** Uses faster-whisper for word-level timestamps
3. **Matching:** Matches script segments against your local `clip_index.json`
4. **Images:** Uses Stable Diffusion XL to generate fallbacks for missing clips.

In [2]:
import os
from pathlib import Path

INPUT_DIR = Path('/content/inputs')
AUDIO_DIR = Path('/content/audio')
CAPTIONS_DIR = Path('/content/captions')
IMAGES_DIR = Path('/content/images')
OUTPUT_DIR = Path('/content/output')

for d in [INPUT_DIR, AUDIO_DIR, CAPTIONS_DIR, IMAGES_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories created!\n')
print('ACTION REQUIRED: Please open the files tab on the left and upload your:')
print('1. script.txt to /content/inputs/')
print('2. clip_index.json to /content/inputs/')
print('3. (Optional) reference.wav and reference.txt for voice cloning to /content/inputs/')

Directories created!

ACTION REQUIRED: Please open the files tab on the left and upload your:
1. script.txt to /content/inputs/
2. clip_index.json to /content/inputs/
3. (Optional) reference.wav and reference.txt for voice cloning to /content/inputs/


In [4]:
# 1. Install Dependencies
!apt-get install -y portaudio19-dev libportaudio2  # Required for pyaudio wheel
!pip install -q faster-whisper diffusers transformers accelerate safetensors huggingface_hub pyaudio

# Install Fish Speech from GitHub
!git clone https://github.com/fishaudio/fish-speech.git
%cd fish-speech
!pip install -q -e .

# Download the Fish Speech 1.4 Checkpoints
# !huggingface-cli download fishaudio/fish-speech-1.4 --local-dir checkpoints/fishaudio/fish-speech-1.4
!huggingface-cli download fishaudio/fish-speech-1.5 --local-dir checkpoints/fishaudio/fish-speech-1.5
!pip install -q -U torchvision torchaudio
!pip install git+https://github.com/huggingface/transformers.git

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libportaudio2 is already the newest version (19.6.0-1.1).
portaudio19-dev is already the newest version (19.6.0-1.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.25.0 requires protobuf!=4.21.0,!=5.28.0,<7,>=3.19.0; python_version > "3.9" and sys_platform == "linux", but you have protobuf 7.35.1 which is incompatible.
descript-audiotools 0.7.2 requires protobuf<3.20,>=3.9.2, but you have protobuf 7.35.1 which is incompatible.
fish-speech 2.0.0 requires torch==2.8.0, but you have torch 2.12.1 which is incompatible.
fish-speech 2.0.0 requires torchaudio==2.8.0, but you have torchaudio 2.11.0 which is incompatible.
f

In [5]:
import subprocess
import shutil
import glob
from pathlib import Path

%cd /content/fish-speech
CHECKPOINT_PATH = "/content/checkpoints/fishaudio/fish-speech-1.5"

# Auto-detect the exact DAC checkpoint file
dac_ckpts = glob.glob(f"{CHECKPOINT_PATH}/*generator*.pth") + glob.glob(f"{CHECKPOINT_PATH}/*codec*.pth")
DAC_CHECKPOINT = dac_ckpts[0] if dac_ckpts else f"{CHECKPOINT_PATH}/firefly-gan-vq-fsq-8x1024-21hz-generator.pth"
print(f"Using DAC Checkpoint: {DAC_CHECKPOINT}")

# Check if reference audio exists for voice cloning
ref_audio = INPUT_DIR / 'reference.wav'
ref_text = INPUT_DIR / 'reference.txt'

use_cloning = ref_audio.exists() and ref_text.exists()
if use_cloning:
    print(f"\nVoice cloning enabled! Extracting VQ tokens from {ref_audio.name}...")
    ref_text_content = ref_text.read_text().strip().replace('"', '\\"')
    cmd0 = f'python fish_speech/models/dac/inference.py -i "{ref_audio}" --checkpoint-path "{DAC_CHECKPOINT}"'
    subprocess.run(cmd0, shell=True)

for txt_file in INPUT_DIR.glob('*.txt'):
    if txt_file.name == 'reference.txt':
        continue

    text = txt_file.read_text().strip().replace('"', '\\"')
    if not text:
        continue

    out_wav = AUDIO_DIR / f"{txt_file.stem}.wav"
    print(f"\nGenerating audio for {txt_file.name}...")

    # Step 1: Text to Semantic Tokens (Takes folder path)
    print("  -> Generating semantic tokens...")
    cmd1 = f'python fish_speech/models/text2semantic/inference.py --text "{text}" --checkpoint-path "{CHECKPOINT_PATH}" --half'
    if use_cloning:
        cmd1 += f' --prompt-text "{ref_text_content}" --prompt-tokens "fake.npy"'

    res1 = subprocess.run(cmd1, shell=True, capture_output=True, text=True)
    if res1.returncode != 0:
        cmd1 = cmd1.replace(" --half", "")
        res1 = subprocess.run(cmd1, shell=True, capture_output=True, text=True)
        if res1.returncode != 0:
            print(f"❌ ERROR in text2semantic:\n{res1.stderr}")
            continue

    # Step 2: Semantic Tokens to Audio (Takes EXACT file path)
    print("  -> Decoding audio from tokens...")
    cmd2 = f'python fish_speech/models/dac/inference.py -i "output/codes_0.npy" --checkpoint-path "{DAC_CHECKPOINT}"'
    res2 = subprocess.run(cmd2, shell=True, capture_output=True, text=True)
    if res2.returncode != 0:
        print(f"❌ ERROR in dac decoder:\n{res2.stderr}")
        continue

    # Move generated file
    if Path("fake.wav").exists():
        shutil.move("fake.wav", out_wav)
        print(f"✅ Audio saved to {out_wav}")
    else:
        print("❌ ERROR: fake.wav was not generated by decoder!")

%cd /content

/content/fish-speech
Using DAC Checkpoint: /content/checkpoints/fishaudio/fish-speech-1.5/firefly-gan-vq-fsq-8x1024-21hz-generator.pth

Generating audio for evil_mortys_master_plan_1.txt...
  -> Generating semantic tokens...
❌ ERROR in text2semantic:
2026-06-21 14:04:29.684 | INFO     | __main__:main:875 - Loading model ...
[transformers] You are using a model of type `dual_ar` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
2026-06-21 14:04:29.687 | WARNING  | fish_speech.models.text2semantic.llama:from_pretrained:508 - Failed to load tokenizer for config injection: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert 

In [8]:
# 2. Voice Generation (Fish Speech v1.4 Pipeline)
import subprocess
import shutil

%cd /content/fish-speech
CHECKPOINT_PATH = "checkpoints/fishaudio/fish-speech-1.4"

# Check if reference audio exists for voice cloning
ref_audio = INPUT_DIR / 'reference.wav'
ref_text = INPUT_DIR / 'reference.txt'

use_cloning = ref_audio.exists() and ref_text.exists()
if use_cloning:
    print(f"\nVoice cloning enabled! Extracting VQ tokens from {ref_audio.name}...")
    ref_text_content = ref_text.read_text().strip().replace('"', '\\"')
    # Step 0: Extract VQ Tokens from reference
    cmd0 = f'python fish_speech/models/dac/inference.py -i "{ref_audio}" --checkpoint-path "{CHECKPOINT_PATH}"'
    res0 = subprocess.run(cmd0, shell=True, capture_output=True, text=True)
    if res0.returncode != 0:
        print(f"\u274c ERROR extracting tokens:\n{res0.stderr}")
else:
    print("\nNo reference.wav found. Using default Fish Speech voice.")

for txt_file in INPUT_DIR.glob('*.txt'):
    if txt_file.name == 'reference.txt':
        continue

    text = txt_file.read_text().strip().replace('"', '\\"')
    if not text:
        continue

    out_wav = AUDIO_DIR / f"{txt_file.stem}.wav"
    print(f"\nGenerating audio for {txt_file.name}...")

    # Step 1: Text to Semantic Tokens
    print("  -> Generating semantic tokens...")
    cmd1 = f'python fish_speech/models/text2semantic/inference.py --text "{text}" --checkpoint-path "{CHECKPOINT_PATH}"'
    if use_cloning:
        cmd1 += f' --prompt-text "{ref_text_content}" --prompt-tokens "fake.npy"'

    res1 = subprocess.run(cmd1, shell=True, capture_output=True, text=True)
    if res1.returncode != 0:
        print(f"\u274c ERROR in text2semantic:\n{res1.stderr}")
        continue

    # Step 2: Semantic Tokens to Audio
    print("  -> Decoding audio from tokens...")
    cmd2 = f'python fish_speech/models/dac/inference.py -i "codes_0.npy" --checkpoint-path "{CHECKPOINT_PATH}"'
    res2 = subprocess.run(cmd2, shell=True, capture_output=True, text=True)
    if res2.returncode != 0:
        print(f"\u274c ERROR in dac decoder:\n{res2.stderr}")
        continue

    # Move generated file to output directory
    if Path("fake.wav").exists():
        shutil.move("fake.wav", out_wav)
        print(f"\u2705 Audio saved to {out_wav}")
    else:
        print("\u274c ERROR: fake.wav was not generated by decoder!")

%cd /content

/content/fish-speech

No reference.wav found. Using default Fish Speech voice.

Generating audio for evil_mortys_master_plan_1.txt...
  -> Generating semantic tokens...
  -> Decoding audio from tokens...
❌ ERROR in dac decoder:
Usage: inference.py [OPTIONS]
Try 'inference.py --help' for help.

Error: Invalid value for '--input-path' / '-i': Path 'codes_0.npy' does not exist.

/content


In [15]:
%%bash
cd /content/fish-speech
python fish_speech/models/text2semantic/inference.py --text "Test audio generation" --checkpoint-path "checkpoints/fishaudio/fish-speech-1.4"
echo "--- FILES GENERATED ---"
ls *.npy

<|im_start|>system
convert the provided text to speech<|im_end|>
<|im_start|>user
Test audio generation<|im_end|>
<|im_start|>assistant
<|voice|>
--- FILES GENERATED ---


2026-06-21 13:37:46.318 | INFO     | __main__:main:875 - Loading model ...
CRITICAL ERROR: No semantic tokens found in vocab! Audio cannot be synthesized.
2026-06-21 13:37:46.399 | INFO     | fish_speech.models.text2semantic.llama:from_pretrained:504 - Injected Semantic IDs into Config: 0-0
2026-06-21 13:37:46.399 | INFO     | fish_speech.models.text2semantic.llama:from_pretrained:520 - Loading model from checkpoints/fishaudio/fish-speech-1.4, config: DualARModelArgs(model_type='dual_ar', vocab_size=32000, n_layer=24, n_head=16, dim=1024, intermediate_size=4096, n_local_heads=2, head_dim=64, rope_base=1000000.0, norm_eps=1e-06, max_seq_len=4096, dropout=0.1, tie_word_embeddings=False, attention_qkv_bias=False, attention_o_bias=False, attention_qk_norm=False, codebook_size=1024, num_codebooks=8, semantic_begin_id=0, semantic_end_id=0, use_gradient_checkpointing=True, initializer_range=0.02, is_reward_model=False, scale_codebook_embeddings=False, audio_embed_dim=None, n_fast_layer=4, fas

CalledProcessError: Command 'b'cd /content/fish-speech\npython fish_speech/models/text2semantic/inference.py --text "Test audio generation" --checkpoint-path "checkpoints/fishaudio/fish-speech-1.4"\necho "--- FILES GENERATED ---"\nls *.npy\n'' returned non-zero exit status 2.

In [14]:
# %%bash
!cd /content/fish-speech
!python fish_speech/models/text2semantic/inference.py --text "Test audio generation" --checkpoint-path "checkpoints/fishaudio/fish-speech-1.4"

python3: can't open file '/content/fish_speech/models/text2semantic/inference.py': [Errno 2] No such file or directory


In [11]:
!ls *.npy

ls: cannot access '*.npy': No such file or directory


In [ ]:
# 3. Captioning (Faster-Whisper)
from faster_whisper import WhisperModel
import json

print('Loading Whisper...')
model = WhisperModel('small', device='cuda', compute_type='float16')

for wav_file in AUDIO_DIR.glob('*.wav'):
    print(f'Captioning {wav_file.name}...')
    segments, info = model.transcribe(str(wav_file), word_timestamps=True)

    cap_data = {'audio_file': str(wav_file), 'segments': []}
    for i, seg in enumerate(segments):
        words = [{'word': w.word, 'start': w.start, 'end': w.end} for w in seg.words]
        cap_data['segments'].append({
            'id': i,
            'text': seg.text.strip(),
            'start': seg.start,
            'end': seg.end,
            'words': words
        })

    out_cap = CAPTIONS_DIR / f"{wav_file.stem}.json"
    with open(out_cap, 'w') as f:
        json.dump(cap_data, f, indent=2)
    print(f"Captions saved to {out_cap}")

In [ ]:
# 4. Matching
import re

clip_index_path = INPUT_DIR / 'clip_index.json'
clips = []
if clip_index_path.exists():
    clips = json.loads(clip_index_path.read_text()).get('clips', [])
    print(f"Loaded {len(clips)} clips from clip_index.json")
else:
    print("No clip_index.json found. All segments will use AI images.")

def extract_keywords(text):
    words = re.sub(r'[^a-z0-9\s]', '', text.lower()).split()
    stops = {'the', 'is', 'at', 'which', 'on', 'in', 'a', 'an', 'of', 'to', 'and', 'or', 'for'}
    return {w for w in words if w not in stops and len(w) > 1}

for cap_file in CAPTIONS_DIR.glob('*.json'):
    cap_data = json.loads(cap_file.read_text())
    manifest = {'audio_file': cap_data['audio_file'], 'segments': []}

    for seg in cap_data['segments']:
        best_clip = None
        best_score = 0
        seg_kw = extract_keywords(seg['text'])

        for c in clips:
            clip_kw = extract_keywords(c.get('action', '')) | set(c.get('tags', []))
            score = len(seg_kw & clip_kw)
            if score > best_score:
                best_score = score
                best_clip = c

        entry = {
            'id': seg['id'],
            'text': seg['text'],
            'start': seg['start'],
            'end': seg['end'],
            'words': seg['words']
        }

        if best_clip and best_score > 0:
            entry['visual_type'] = 'clip'
            entry['visual_source'] = best_clip['filename']
            entry['clip_start'] = 0.0
        else:
            entry['visual_type'] = 'ai_image'
            clean_text = seg['text'][:100].replace('"', '').replace("'", '')
            entry['visual_source'] = f"Cinematic still depicting: {clean_text}. Dramatic lighting, high detail, 9:16 vertical composition."

        manifest['segments'].append(entry)

    out_manifest = OUTPUT_DIR / f"manifest_{cap_file.stem}.json"
    with open(out_manifest, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f"Manifest saved to {out_manifest}")

In [ ]:
# 5. Image Generation (SDXL)
import torch
from diffusers import DiffusionPipeline

print('Loading Stable Diffusion XL...')
pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant='fp16'
)
pipe.to('cuda')

for man_file in OUTPUT_DIR.glob('manifest_*.json'):
    man_data = json.loads(man_file.read_text())
    for seg in man_data['segments']:
        if seg.get('visual_type') == 'ai_image':
            prompt = seg['visual_source']
            out_img = IMAGES_DIR / f"seg_{seg['id']:04d}.png"

            if not out_img.exists():
                print(f'Generating image for segment {seg["id"]}...')
                image = pipe(
                    prompt=prompt,
                    width=1080,
                    height=1920,
                    num_inference_steps=25
                ).images[0]
                image.save(out_img)
            else:
                print(f'Image {out_img.name} already exists. Skipping.')

In [ ]:
# 6. Package and Zip Outputs
import shutil

print('Zipping outputs...')
# Zip the whole content directory except the inputs and huge repos to save time
os.makedirs('/content/zip_staging', exist_ok=True)
for folder in ['audio', 'captions', 'images', 'output']:
    src = f'/content/{folder}'
    dst = f'/content/zip_staging/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)

shutil.make_archive('/content/explainer_outputs', 'zip', '/content/zip_staging')
print('\n\u2705 Done! Please download /content/explainer_outputs.zip')

# Optional: Automatically trigger download in browser
try:
    from google.colab import files
    files.download('/content/explainer_outputs.zip')
except ImportError:
    pass